In [2]:
import pandas as pd
trades = pd.read_csv("DATA/historical_data.csv")

sentiment = pd.read_csv("DATA/fear_greed_index.csv")
print("Trades Shape",trades.shape)
print("Sentiment Shape",sentiment.shape)

Trades Shape (211224, 16)
Sentiment Shape (2644, 4)


In [3]:
trades.columns


Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='str')

In [4]:
sentiment.columns

Index(['timestamp', 'value', 'classification', 'date'], dtype='str')

In [5]:
trades.isnull().sum()


Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [6]:
trades.duplicated().sum()

np.int64(0)

In [7]:
# Convert trade timestamp to datetime
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'],dayfirst=True )

# Create date column (daily level)
trades['date'] = trades['Timestamp IST'].dt.date

# Convert sentiment date
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

trades[['Timestamp IST','date']].head()

,Timestamp IST,date
0,2024-12-02 22:50:00,2024-12-02
1,2024-12-02 22:50:00,2024-12-02
2,2024-12-02 22:50:00,2024-12-02
3,2024-12-02 22:50:00,2024-12-02
4,2024-12-02 22:50:00,2024-12-02


In [8]:
merged =pd.merge(
    trades,sentiment[["date","classification"]],
    on="date",
    how="inner"
)

print("Merged Shape:", merged.shape)

Merged Shape: (211218, 18)


In [9]:
merged.groupby("classification")["Closed PnL"].mean()

classification
Extreme Fear     34.537862
Extreme Greed    67.892861
Fear             54.290400
Greed            42.743559
Neutral          34.307718
Name: Closed PnL, dtype: float64

In [10]:
merged["win"] = merged["Closed PnL"]>0
merged.groupby("classification")["win"].mean()

classification
Extreme Fear     0.370607
Extreme Greed    0.464943
Fear             0.420768
Greed            0.384828
Neutral          0.396991
Name: win, dtype: float64

In [11]:
merged["classification"].value_counts()

classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64

In [12]:
merged.groupby("classification")["Start Position"].mean()

classification
Extreme Fear      -2322.304587
Extreme Greed     19518.990812
Fear               8709.824822
Greed           -151088.546635
Neutral             144.002263
Name: Start Position, dtype: float64

In [13]:
merged.groupby(["classification","Side"]).size().unstack()

Side,BUY,SELL
classification,,
Extreme Fear,10935,10465
Extreme Greed,17940,22052
Fear,30270,31567
Greed,24576,25727
Neutral,18969,18717


In [14]:
merged.groupby("classification")["Size USD"].mean()

classification
Extreme Fear     5349.731843
Extreme Greed    3112.251565
Fear             7816.109931
Greed            5736.884375
Neutral          4782.732661
Name: Size USD, dtype: float64

In [15]:
merged.groupby('classification')['Closed PnL'].std()

classification
Extreme Fear     1136.056091
Extreme Greed     766.828294
Fear              935.355438
Greed            1116.028390
Neutral           517.122220
Name: Closed PnL, dtype: float64

In [16]:
trade_counts = merged.groupby('Account').size().reset_index(name='trade_count')

trade_counts.head()

,Account,trade_count
0,0x083384f897ee0f19899168e3b1bec365f52a9012,3818
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,7280
2,0x271b280974205ca63b716753467d5a371de622ab,3809
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,13311
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3239


In [17]:
threshold = trade_counts['trade_count'].quantile(0.75)

trade_counts['segment_freq'] = trade_counts['trade_count'].apply(
    lambda x: 'Frequent' if x >= threshold else 'Infrequent'
)

trade_counts['segment_freq'].value_counts()

segment_freq
Infrequent    24
Frequent       8
Name: count, dtype: int64

In [18]:
trade_counts.head()

,Account,trade_count,segment_freq
0,0x083384f897ee0f19899168e3b1bec365f52a9012,3818,Infrequent
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,7280,Infrequent
2,0x271b280974205ca63b716753467d5a371de622ab,3809,Infrequent
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,13311,Frequent
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3239,Infrequent


In [19]:
merged.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'date', 'classification', 'win'],
      dtype='str')

In [20]:
account_pnl = merged.groupby('Account')['Closed PnL'].mean().reset_index()

threshold = account_pnl['Closed PnL'].quantile(0.75)

account_pnl['segment_profit'] = account_pnl['Closed PnL'].apply(
    lambda x: 'High Profit' if x >= threshold else 'Others'
)

account_pnl['segment_profit'].value_counts()

segment_profit
Others         24
High Profit     8
Name: count, dtype: int64

In [21]:
merged = merged.drop(columns=['segment_profit'], errors='ignore')

In [22]:
merged = merged.merge(
    account_pnl[['Account','segment_profit']],
    on='Account',
    how='left'
)

In [23]:
merged = merged.drop(columns=['segment_profit'], errors='ignore')

In [24]:
merged = merged.merge(
    account_pnl[['Account','segment_profit']],
    on='Account',
    how='left'
)

In [25]:
print('segment_profit' in merged.columns)

True


In [26]:
merged.groupby('segment_profit')['Closed PnL'].mean()

segment_profit
High Profit    281.054964
Others          32.602409
Name: Closed PnL, dtype: float64

In [27]:
merged.groupby('segment_profit')['win'].mean()

segment_profit
High Profit    0.410046
Others         0.411331
Name: win, dtype: float64

In [28]:
import matplotlib.pyplot as plt

In [29]:
import sys
!{sys.executable} -m pip install matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# import matplotlib.pyplot as plt

# sentiment_pnl = merged.groupby('classification')['Closed PnL'].mean()

# plt.figure()
# sentiment_pnl.plot(kind='bar')
# plt.title("Average PnL by Market Sentiment")
# plt.xlabel("Sentiment")
# plt.ylabel("Average Closed PnL")
# plt.xticks(rotation=45)
# plt.show()

In [31]:
# profit_pnl = merged.groupby('segment_profit')['Closed PnL'].mean()

# plt.figure()
# profit_pnl.plot(kind='bar')
# plt.title("Average PnL: High Profit vs Others")
# plt.xlabel("Trader Segment")
# plt.ylabel("Average Closed PnL")
# plt.show()

In [32]:
# win_rate = merged.groupby('segment_profit')['win'].mean()

# plt.figure()
# win_rate.plot(kind='bar')
# plt.title("Win Rate Comparison")
# plt.xlabel("Trader Segment")
# plt.ylabel("Win Rate")
# plt.show()